In [1]:
import xarray as xr
from netCDF4 import Dataset
import cfgrib
import eccodes
import pandas as pd
import numpy as np

In [13]:
import re
from pathlib import Path

def convert_grib2nc_pl(f, overwrite=False):
    pl_pattern = re.compile(r"era5_pl_(\d{8}_to_\d{8}).grib")
    new_file = pl_pattern.sub(r"era5_pl_\1.nc", f)
    if not overwrite and Path(new_file).exists():
        print(f"Skipping {new_file}")
        return
    print(f"Converting {Path(f).name}")
    print(new_file)
    print('Opening GRIB dataset')
    ds = xr.open_dataset(f, engine="cfgrib", indexpath='', decode_timedelta=False)
    print('Dropping unused vraiables')
    ds = ds.drop_vars(['step', 'valid_time'])
    print('Renaming dimensions')
    ds = ds.rename_dims({'time':'valid_time', 'isobaricInhPa': 'pressure_level'})
    print('Renaming variables')
    ds = ds.rename_vars({'time':'valid_time', 'isobaricInhPa': 'pressure_level'})
    print("Transposing dimensions in the format: ['latitude', 'longitude', 'pressure_level', 'valid_time']")
    ds = ds.transpose('latitude', 'longitude', 'pressure_level', 'valid_time')
    print("Creating 'expver' variable")
    ds = ds.assign_coords(expver=("valid_time", ['0001']*len(ds.valid_time)))
    print('Saving to netCDF4 file')
    ds.load().to_netcdf(new_file)
    print(f"Done converting {Path(f).name}")

In [14]:
convert_grib2nc_pl('/fs/yedoma/data/globsim/n60_test_grib/era5/era5_pl_20250201_to_20250231.grib')

Converting era5_pl_20250201_to_20250231.grib
/fs/yedoma/data/globsim/n60_test_grib/era5/era5_pl_20250201_to_20250231.nc
Opening GRIB dataset
Dropping unused vraiables
Renaming dimensions
Renaming variables
Transposing dimensions in the format: ['latitude', 'longitude', 'pressure_level', 'valid_time']
Creating 'expver' variable
Saving to netCDF4 file
Done converting era5_pl_20250201_to_20250231.grib


In [33]:
def convert_grib2nc_to(file):
    out_name_temp = f'{file.split('.grib')[0]}_test_temp.nc'
    out_name = f'{file.split('.grib')[0]}_test.nc'
    ! grib_to_netcdf -o {out_name_temp} {file}
    # ds = xr.open_dataset(file, engine="cfgrib", indexpath='')
    # ds = ds.drop_vars(['step', 'surface', 'time'])
    # ds = ds.expand_dims('valid_time', create_index_for_new_dim=True)
    ds = xr.open_dataset(out_name_temp, engine='netcdf4')
    ds = ds.assign_coords(expver=('valid_time', ['0001']))
    ds = ds.transpose('latitude', 'longitude', 'valid_time')
    ds.to_netcdf(out_name)

In [48]:
file = '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to.grib'
out_name = f'{file.split('.grib')[0]}_test.nc'
! grib_to_netcdf -o {out_name} {file}
ds = xr.open_dataset(out_name, engine='netcdf4')
ds = ds.rename_dims({'time': 'valid_time'})
ds = ds.rename_vars({'time': 'valid_time'})
ds = ds.assign_coords(number=('valid_time', [0]))
ds = ds.assign_coords(expver=('valid_time', ['0001']))
ds = ds.transpose('latitude', 'longitude', 'valid_time')
ds.to_netcdf(out_name)

grib_to_netcdf: Version 2.41.0
grib_to_netcdf: Processing input file '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to.grib'.
grib_to_netcdf: Found 2 GRIB fields in 1 file.
grib_to_netcdf: Ignoring key(s): method, type, stream, refdate, hdate
grib_to_netcdf: Creating netCDF file '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to_test.nc'
grib_to_netcdf: NetCDF library version: 4.9.2 of Aug 30 2023 18:15:22 $
grib_to_netcdf: Creating large (64 bit) file format.
grib_to_netcdf: Defining variable 'lsm'.
grib_to_netcdf: Defining variable 'z'.
grib_to_netcdf: Done.


In [55]:
ds_grib

<xarray.Dataset> Size: 76kB
Dimensions:     (latitude: 45, longitude: 205)
Coordinates:
    number      int64 8B 0
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * latitude    (latitude) float64 360B 71.0 70.75 70.5 ... 60.5 60.25 60.0
  * longitude   (longitude) float64 2kB -141.0 -140.8 -140.5 ... -90.25 -90.0
    valid_time  datetime64[ns] 8B 2000-01-01
Data variables:
    lsm         (latitude, longitude) float32 37kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    z           (latitude, longitude) float32 37kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-05-07T18:46 GRIB to CDM+CF via cfgrib-0.9.1...

In [53]:
print(np.all(ds_grib.number.values == ds.number.values))
print(np.all(ds_grib.valid_time.values == ds.valid_time.values))
print(np.all(ds_grib.latitude.values == ds.latitude.values))
print(np.all(ds_grib.longitude.values == ds.longitude.values))
print(np.all(ds_grib.expver.values == ds.expver.values))

True
True
True
True


AttributeError: 'Dataset' object has no attribute 'expver'

In [51]:
ds_grib = xr.open_dataset(file, engine="cfgrib", indexpath='')

/fs/yedoma/home/vpo001/globsim/.conda/lib/python3.12/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [52]:
ds_grib.lsm.values

array([[0.        , 0.        , 0.        , ..., 0.        , 0.02439129,
        0.07754779],
       [0.        , 0.        , 0.        , ..., 0.        , 0.03227878,
        0.0436995 ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [1.        , 1.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       [0.9994534 , 0.99994934, 0.999949  , ..., 0.        , 0.        ,
        0.        ],
       [0.8431269 , 0.9994    , 0.99897885, ..., 0.        , 0.        ,
        0.        ]], shape=(45, 205), dtype=float32)

In [34]:
convert_grib2nc_to('/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to.grib')

grib_to_netcdf: Version 2.41.0
grib_to_netcdf: Processing input file '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to.grib'.
grib_to_netcdf: Found 2 GRIB fields in 1 file.
grib_to_netcdf: Ignoring key(s): method, type, stream, refdate, hdate
grib_to_netcdf: Creating netCDF file '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to_test_temp.nc'
grib_to_netcdf: NetCDF library version: 4.9.2 of Aug 30 2023 18:15:22 $
grib_to_netcdf: Creating large (64 bit) file format.
grib_to_netcdf: Defining variable 'lsm'.
grib_to_netcdf: Defining variable 'z'.
grib_to_netcdf: Done.


ValueError: ('latitude', 'longitude', 'valid_time') must be a permuted list of FrozenMappingWarningOnValuesAccess({'longitude': 205, 'latitude': 45, 'time': 1, 'valid_time': 1}), unless `...` is included

In [140]:
def convert_grib2nc_to(file):
    ds = xr.open_dataset(file, engine="cfgrib", indexpath='')
    ds = ds.drop_vars(['step', 'surface', 'time'])
    ds = ds.expand_dims('valid_time', create_index_for_new_dim=True)
    ds = ds.assign_coords(expver=('valid_time', ['0001']))
    ds = ds.transpose('latitude', 'longitude', 'valid_time')
    ds.to_netcdf(f'{file.split('.grib')[0]}.nc')

In [141]:
convert_grib2nc_to('/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to.grib')

/fs/yedoma/home/vpo001/globsim/.conda/lib/python3.12/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [142]:
def convert_grib2nc_pl(file):
    ds = xr.open_dataset(file, engine="cfgrib", indexpath='')
    ds = ds.drop_vars(['step', 'valid_time'])
    ds = ds.rename_dims({'time':'valid_time', 'isobaricInhPa': 'pressure_level'})
    ds = ds.rename_vars({'time':'valid_time', 'isobaricInhPa': 'pressure_level'})
    ds = ds.transpose('latitude', 'longitude', 'pressure_level', 'valid_time')
    ds = ds.assign_coords(expver=("valid_time", ['0001']*len(ds.valid_time)))
    ds.to_netcdf(f'{file.split('.grib')[0]}.nc')

In [ ]:
convert_grib2nc_pl('/fs/yedoma/data/globsim/n60_test_grib/era5/era5_pl_19400101_to_19400131.grib')

In [84]:
ds = xr.open_dataset('/fs/yedoma/data/globsim/n60_test_grib/era5/era5_pl_19400101_to_19400131.grib', engine="cfgrib", indexpath='')
ds = ds.drop_vars(['step', 'valid_time'])
ds = ds.rename_dims({'time':'valid_time', 'isobaricInhPa': 'pressure_level'})
ds = ds.rename_vars({'time':'valid_time', 'isobaricInhPa': 'pressure_level'})
ds = ds.transpose('latitude', 'longitude', 'pressure_level', 'valid_time')
ds = ds.assign_coords(expver=("valid_time", ['0001']*len(ds.valid_time)))

/fs/yedoma/home/vpo001/globsim/.conda/lib/python3.12/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [85]:
ds

<xarray.Dataset> Size: 2GB
Dimensions:         (valid_time: 744, pressure_level: 15, latitude: 45,
                     longitude: 205)
Coordinates:
    number          int64 8B ...
  * valid_time      (valid_time) datetime64[ns] 6kB 1940-01-01 ... 1940-01-31...
  * pressure_level  (pressure_level) float64 120B 1e+03 975.0 ... 600.0 550.0
  * latitude        (latitude) float64 360B 71.0 70.75 70.5 ... 60.5 60.25 60.0
  * longitude       (longitude) float64 2kB -141.0 -140.8 ... -90.25 -90.0
    expver          (valid_time) <U4 12kB '0001' '0001' '0001' ... '0001' '0001'
Data variables:
    t               (latitude, longitude, pressure_level, valid_time) float32 412MB ...
    r               (latitude, longitude, pressure_level, valid_time) float32 412MB ...
    u               (latitude, longitude, pressure_level, valid_time) float32 412MB ...
    v               (latitude, longitude, pressure_level, valid_time) float32 412MB ...
    z               (latitude, longitude, pressure_level, valid_time) float32 412MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-05-06T20:37 GRIB to CDM+CF via cfgrib-0.9.1...

In [56]:
ds.rename_vars({'isobaricInhPa': 'pressure_level'})

<xarray.Dataset> Size: 2GB
Dimensions:         (time: 744, isobaricInhPa: 15, latitude: 45, longitude: 205)
Coordinates:
    number          int64 8B ...
  * time            (time) datetime64[ns] 6kB 1940-01-01 ... 1940-01-31T23:00:00
  * pressure_level  (isobaricInhPa) float64 120B 1e+03 975.0 ... 600.0 550.0
  * latitude        (latitude) float64 360B 71.0 70.75 70.5 ... 60.5 60.25 60.0
  * longitude       (longitude) float64 2kB -141.0 -140.8 ... -90.25 -90.0
    valid_time      (time) datetime64[ns] 6kB ...
Dimensions without coordinates: isobaricInhPa
Data variables:
    t               (time, isobaricInhPa, latitude, longitude) float32 412MB ...
    r               (time, isobaricInhPa, latitude, longitude) float32 412MB ...
    u               (time, isobaricInhPa, latitude, longitude) float32 412MB ...
    v               (time, isobaricInhPa, latitude, longitude) float32 412MB ...
    z               (time, isobaricInhPa, latitude, longitude) float32 412MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-05-06T20:01 GRIB to CDM+CF via cfgrib-0.9.1...

In [83]:
ds.to_netcdf('/fs/yedoma/data/globsim/n60_test_grib/era5/era5_pl_19400101_to_19400131.nc')

In [8]:
import pygrib
grbs = pygrib.open('/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.grib')

In [16]:
grbs.seek(0)
for grb in grbs:
    print(grb)

1:Total precipitation:m (accum):regular_ll:surface:level 0:fcst time 5-6 hrs (accum):from 194001311800
2:Surface short-wave (solar) radiation downwards:J m**-2 (accum):regular_ll:surface:level 0:fcst time 5-6 hrs (accum):from 194001311800
3:Surface long-wave (thermal) radiation downwards:J m**-2 (accum):regular_ll:surface:level 0:fcst time 5-6 hrs (accum):from 194001311800
4:2 metre temperature:K (instant):regular_ll:surface:level 0:fcst time 0 hrs:from 194002010000
5:2 metre dewpoint temperature:K (instant):regular_ll:surface:level 0:fcst time 0 hrs:from 194002010000
6:10 metre U wind component:m s**-1 (instant):regular_ll:surface:level 0:fcst time 0 hrs:from 194002010000
7:10 metre V wind component:m s**-1 (instant):regular_ll:surface:level 0:fcst time 0 hrs:from 194002010000
8:Total column ozone:kg m**-2 (instant):regular_ll:surface:level 0:fcst time 0 hrs:from 194002010000
9:Total column vertically-integrated water vapour:kg m**-2 (instant):regular_ll:surface:level 0:fcst time 0 hr

In [22]:
! grib_to_netcdf -o '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to_test.nc' '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to.grib'

grib_to_netcdf: Version 2.41.0
grib_to_netcdf: Processing input file '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to.grib'.
grib_to_netcdf: Found 2 GRIB fields in 1 file.
grib_to_netcdf: Ignoring key(s): method, type, stream, refdate, hdate
grib_to_netcdf: Creating netCDF file '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_to_test.nc'
grib_to_netcdf: NetCDF library version: 4.9.2 of Aug 30 2023 18:15:22 $
grib_to_netcdf: Creating large (64 bit) file format.
grib_to_netcdf: Defining variable 'lsm'.
grib_to_netcdf: Defining variable 'z'.
grib_to_netcdf: Done.


In [12]:
new_file = '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.nc'

! cdo -f nc -copy '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.grib' {new_file}
# grib_file = '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.grib'
with xr.open_dataset(new_file, engine='netcdf4') as ds:
        ds = ds.rename_dims({'time': 'valid_time', 'lat': 'latitude', 'lon': 'longitude'})
        ds = ds.rename_vars({'time': 'valid_time', 'lat': 'latitude', 'lon': 'longitude'}) 
        # ds = ds.transpose('latitude', 'longitude', 'valid_time')
        # ds_grib = xr.open_dataset(grib_file, engine="cfgrib", indexpath='')
        # for var in list(ds_grib.var()):
        #         ds[var].values = ds_grib[var].values
        ds.to_netcdf(new_file)

cdo copy/selall : UNCHANGED_RECORD=0
cdo copy/selall : cdiGribDataScanningMode=0; lcopy=0
cdo copy: Processed 57785400 values from 9 variables over 696 timesteps [6.91s 140MB]


In [67]:
! grib_to_netcdf -o '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.nc' '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.grib'

grib_to_netcdf: Version 2.41.0
grib_to_netcdf: Processing input file '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.grib'.
grib_to_netcdf: Found 6264 GRIB fields in 1 file.
grib_to_netcdf: Ignoring key(s): method, type, stream, refdate, hdate
grib_to_netcdf: Creating netCDF file '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.nc'
grib_to_netcdf: NetCDF library version: 4.9.2 of Aug 30 2023 18:15:22 $
grib_to_netcdf: Creating large (64 bit) file format.
grib_to_netcdf: Defining variable 'tp'.
grib_to_netcdf: Defining variable 'ssrd'.
grib_to_netcdf: Defining variable 'strd'.
grib_to_netcdf: Defining variable 't2m'.
grib_to_netcdf: Defining variable 'd2m'.
grib_to_netcdf: Defining variable 'u10'.
grib_to_netcdf: Defining variable 'v10'.
grib_to_netcdf: Defining variable 'tco3'.
grib_to_netcdf: Defining variable 'tcwv'.
grib_to_netcdf: Done.


In [65]:
grib_file = '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.grib'
new_file = '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.nc'
with xr.open_dataset(new_file, engine='netcdf4') as ds:
        ds = ds.rename_dims({'time': 'valid_time'})
        ds = ds.rename_vars({'time': 'valid_time'}) 
        ds = ds.transpose('latitude', 'longitude', 'valid_time')
        ds_grib = xr.open_dataset(grib_file, engine="cfgrib", indexpath='')
        for var in list(ds_grib.var()):
                ds[var].values = ds_grib[var].values
        ds.to_netcdf(new_file)

skipping variable: paramId==167 shortName='t2m'
Traceback (most recent call last):
  File "/fs/yedoma/home/vpo001/globsim/.conda/lib/python3.12/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "/fs/yedoma/home/vpo001/globsim/.conda/lib/python3.12/site-packages/cfgrib/dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='time' value=Variable(dimensions=('time',), data=array([-944114400, -944071200, -944028000, -943984800, -943941600,
       -943898400, -943855200, -943812000, -943768800, -943725600,
       -943682400, -943639200, -943596000, -943552800, -943509600,
       -943466400, -943423200, -943380000, -943336800, -943293600,
       -943250400, -943207200, -943164000, -943120800, -943077600,
       -943034400, -942991200, -942948000, -942904800, -942861600,
       -942818400, -942775200, -942732000, -942688800, -942645600,
  

ValueError: replacement data must match the Variable's shape. replacement data has shape (59, 12, 45, 205); Variable has shape (45, 205, 696)

In [17]:
! nccopy -V time,latitude,longitude,ssrd,strd,tp '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.grib' '/fs/yedoma/data/globsim/n60_test_grib/era5/era5_sf_19400201_to_19400231.grib'

NetCDF: Unknown file format
Location: file ?; fcn ? line 2016


In [19]:
ds = xr.open_dataset('/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.grib', engine="cfgrib",
                     indexpath='')
ds.to_netcdf('/fs/yedoma/data/globsim/n60_test_grib/era5/era5_re_resl_19400201_to_19400231.nc')
# ds = ds.drop_vars(['step', 'valid_time'])
# ds = ds.rename_dims({'time':'valid_time', 'isobaricInhPa': 'pressure_level'})
# ds = ds.rename_vars({'time':'valid_time', 'isobaricInhPa': 'pressure_level'})
# ds = ds.transpose('latitude', 'longitude', 'pressure_level', 'valid_time')
# ds = ds.assign_coords(expver=("valid_time", ['0001']*len(ds.valid_time)))

skipping variable: paramId==167 shortName='t2m'
Traceback (most recent call last):
  File "/fs/yedoma/home/vpo001/globsim/.conda/lib/python3.12/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "/fs/yedoma/home/vpo001/globsim/.conda/lib/python3.12/site-packages/cfgrib/dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='time' value=Variable(dimensions=('time',), data=array([-944114400, -944071200, -944028000, -943984800, -943941600,
       -943898400, -943855200, -943812000, -943768800, -943725600,
       -943682400, -943639200, -943596000, -943552800, -943509600,
       -943466400, -943423200, -943380000, -943336800, -943293600,
       -943250400, -943207200, -943164000, -943120800, -943077600,
       -943034400, -942991200, -942948000, -942904800, -942861600,
       -942818400, -942775200, -942732000, -942688800, -942645600,
  

In [14]:
59*12

708

In [13]:
ds

<xarray.Dataset> Size: 78MB
Dimensions:     (time: 59, step: 12, latitude: 45, longitude: 205)
Coordinates:
    number      int64 8B ...
  * time        (time) datetime64[ns] 472B 1940-01-31T18:00:00 ... 1940-02-29...
  * step        (step) timedelta64[ns] 96B 01:00:00 02:00:00 ... 12:00:00
    surface     float64 8B ...
  * latitude    (latitude) float64 360B 71.0 70.75 70.5 ... 60.5 60.25 60.0
  * longitude   (longitude) float64 2kB -141.0 -140.8 -140.5 ... -90.25 -90.0
    valid_time  (time, step) datetime64[ns] 6kB ...
Data variables:
    tp          (time, step, latitude, longitude) float32 26MB ...
    ssrd        (time, step, latitude, longitude) float32 26MB ...
    strd        (time, step, latitude, longitude) float32 26MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-05-07T17:26 GRIB to CDM+CF via cfgrib-0.9.1...